# 🧠 MSI-Net — Visual Saliency Prediction

**Kiến trúc gốc MSI-Net:**
- Encoder: VGG16-style CNN (conv1→conv5), load **pretrained VGG16-Hybrid weights** (ImageNet + Places)
- **Multi-Scale Input (MSI)**: ảnh được xử lý ở 3 scale (1×, 1/2×, 1/4×) song song, features concat trước decoder
- Decoder: UpSampling2D 8× + Conv 1×1 + sigmoid
- Loss: KL Divergence
- Data: aspect-ratio-preserving resize + symmetric padding
- Training: SALICON → Fine-tune MIT1003 & CAT2000


## ── Bước 1: Cài đặt thư viện ──────────────────────────────


In [ ]:
%%capture
!pip install gdown h5py scipy matplotlib tensorflow==2.12.0 requests

## ── Bước 2: Khởi tạo thư mục làm việc ────────────────────


In [ ]:
import os, sys
BASE_DIR    = "/kaggle/working"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
WEIGHTS_DIR = os.path.join(BASE_DIR, "weights")
for d in [DATA_DIR, RESULTS_DIR, WEIGHTS_DIR,
          os.path.join(RESULTS_DIR, "history"),
          os.path.join(RESULTS_DIR, "images"),
          os.path.join(RESULTS_DIR, "ckpts", "best")]:
    os.makedirs(d, exist_ok=True)
print("✅ Thư mục tạo xong")

## ── Bước 3: Định nghĩa các module (config, loss, model, data, metrics, download) ──


In [ ]:
%%writefile /kaggle/working/config.py
PARAMS = {
    "n_epochs":      10,
    "batch_size":    4,
    "learning_rate": 1e-5,
    "device":        "gpu",
}
DIMS = {
    "image_size_salicon":  (240, 320),
    "image_size_mit1003":  (360, 360),
    "image_size_cat2000":  (216, 384),   # gốc: 1080×1920 scale /5 → 216×384
    "image_size_dutomron": (360, 360),
    "image_size_pascals":  (360, 360),
    "image_size_osie":     (240, 320),
    "image_size_fiwi":     (216, 384),
}


In [ ]:
%%writefile /kaggle/working/loss.py
import tensorflow as tf

def kld(y_true, y_pred, eps=1e-7):
    """KL Divergence loss — khớp với loss.py gốc MSI-Net.
    y_true: saliency map gốc (giá trị 0-255 hoặc 0-1 đều OK vì normalize)
    y_pred: predicted map (0-1)
    """
    sum_per_image = tf.reduce_sum(y_true, axis=(1,2,3), keepdims=True)
    y_true = y_true / (eps + sum_per_image)

    sum_per_image = tf.reduce_sum(y_pred, axis=(1,2,3), keepdims=True)
    y_pred = y_pred / (eps + sum_per_image)

    loss = y_true * tf.math.log(eps + y_true / (eps + y_pred))
    return tf.reduce_mean(tf.reduce_sum(loss, axis=(1,2,3)))


In [ ]:
%%writefile /kaggle/working/model.py
"""
MSI-Net: Multi-Scale Input Network for Visual Saliency Prediction.
Khớp với kiến trúc gốc:
  - Encoder VGG16-style (13 conv layers)
  - Multi-Scale Input: ảnh full + 1/2 + 1/4 được encode riêng, features concat
  - Decoder: UpSampling 8x + Conv1x1 sigmoid
  - Load pretrained VGG16-Hybrid weights (nếu có)
"""
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import os

# ── Tên các layer VGG16 theo thứ tự để map pretrained weights ────────────
VGG_LAYER_NAMES = [
    'conv1_1','conv1_2',
    'conv2_1','conv2_2',
    'conv3_1','conv3_2','conv3_3',
    'conv4_1','conv4_2','conv4_3',
    'conv5_1','conv5_2','conv5_3',
]

def _vgg_block(x, filters, n_convs, block_idx):
    """Một VGG block gồm n_convs conv layers + MaxPool."""
    for i in range(1, n_convs + 1):
        name = f'conv{block_idx}_{i}'
        x = layers.Conv2D(filters, 3, activation='relu', padding='same', name=name)(x)
    x = layers.MaxPooling2D(2, 2, padding='same', name=f'pool{block_idx}')(x)
    return x

def _vgg_encoder(inp, suffix=''):
    """
    VGG16-style encoder (conv1→conv5, không có pool5).
    suffix: dùng để phân biệt tên layer giữa 3 scale ('', '_half', '_quarter')
    """
    def conv(x, f, name):
        return layers.Conv2D(f, 3, activation='relu', padding='same',
                             name=name+suffix)(x)
    def pool(x, name):
        return layers.MaxPooling2D(2, 2, padding='same', name=name+suffix)(x)

    x = conv(inp, 64,  'conv1_1'); x = conv(x, 64,  'conv1_2'); x = pool(x, 'pool1')
    x = conv(x,  128, 'conv2_1'); x = conv(x, 128, 'conv2_2'); x = pool(x, 'pool2')
    x = conv(x,  256, 'conv3_1'); x = conv(x, 256, 'conv3_2')
    x = conv(x,  256, 'conv3_3'); x = pool(x, 'pool3')
    x = conv(x,  512, 'conv4_1'); x = conv(x, 512, 'conv4_2')
    x = conv(x,  512, 'conv4_3'); x = pool(x, 'pool4')
    x = conv(x,  512, 'conv5_1'); x = conv(x, 512, 'conv5_2')
    x = conv(x,  512, 'conv5_3')
    # Không có pool5 — giữ spatial resolution để upsample sau
    return x

def build_msinet(input_shape=(240, 320, 3)):
    """
    Xây dựng MSI-Net với 3 nhánh encoder (full, 1/2, 1/4 scale).
    Feature maps từ 3 nhánh được upsample về cùng kích thước rồi concat.
    Decoder: UpSampling 8× + Conv1x1 sigmoid + Resizing về input size.
    """
    h, w = input_shape[0], input_shape[1]

    # ── 3 input tensors ở 3 scale ──────────────────────────────────────
    inp_full    = tf.keras.Input(shape=input_shape,          name='input_full')
    inp_half    = tf.keras.Input(shape=(h//2, w//2, 3),      name='input_half')
    inp_quarter = tf.keras.Input(shape=(h//4, w//4, 3),      name='input_quarter')

    # ── 3 nhánh encoder ────────────────────────────────────────────────
    feat_full    = _vgg_encoder(inp_full,    suffix='')
    feat_half    = _vgg_encoder(inp_half,    suffix='_half')
    feat_quarter = _vgg_encoder(inp_quarter, suffix='_quarter')

    # ── Upsample half & quarter về cùng spatial size với full ──────────
    # Dùng Lambda lấy shape động từ feat_full để tránh lỗi khi h/w không
    # chia hết cho 16 (vd: 360/16=22.5 → full=23, half=22 → mismatch)
    feat_half_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_half')([feat_half, feat_full])
    feat_quarter_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_quarter')([feat_quarter, feat_full])

    # ── Concat 3 nhánh → 512×3 = 1536 channels ─────────────────────────
    merged = layers.Concatenate(name='concat_scales')(
                [feat_full, feat_half_up, feat_quarter_up])

    # ── Decoder ─────────────────────────────────────────────────────────
    x = layers.UpSampling2D(size=(16, 16), name='upsample')(merged)
    x = layers.Conv2D(1, 1, activation='sigmoid',
                      padding='same', name='output')(x)
    out = layers.Resizing(h, w, name='resize_output')(x)

    model = Model(inputs=[inp_full, inp_half, inp_quarter],
                  outputs=out, name='MSINet')
    return model


def load_vgg16_hybrid_weights(model, weights_path):
    """
    Load pretrained VGG16-Hybrid weights vào các nhánh encoder.
    weights_path: đường dẫn đến file .npy hoặc .h5 chứa VGG16 weights.
    Weights được copy sang cả 3 nhánh (full, half, quarter).
    """
    if not os.path.exists(weights_path):
        print(f'  ⚠️  Không tìm thấy weights: {weights_path} — bỏ qua load pretrained')
        return

    print(f'  Đang load VGG16-Hybrid weights từ {weights_path} ...')
    try:
        w = np.load(weights_path, allow_pickle=True).item()
    except Exception:
        try:
            import h5py
            w = {}
            with h5py.File(weights_path, 'r') as f:
                for k in f.keys():
                    w[k] = [np.array(f[k][wk]) for wk in f[k].keys()]
        except Exception as e:
            print(f'  ⚠️  Không đọc được weights: {e}')
            return

    suffixes = ['', '_half', '_quarter']
    copied = 0
    for name in VGG_LAYER_NAMES:
        if name not in w:
            continue
        for sfx in suffixes:
            layer_name = name + sfx
            try:
                layer = model.get_layer(layer_name)
                layer.set_weights(w[name])
                copied += 1
            except (ValueError, KeyError):
                pass
    print(f'  ✅ Copied pretrained weights vào {copied} layers')


In [ ]:
%%writefile /kaggle/working/data_loader.py
"""
tf.data pipeline cho MSI-Net.
Khớp với data.py gốc:
  - Resize giữ aspect ratio (area khi shrink, bicubic khi enlarge)
  - Symmetric padding (126 cho ảnh RGB, 0 cho saliency)
  - Tạo 3 scale inputs: full, 1/2, 1/4
"""
import os, tensorflow as tf

def _get_file_list(path):
    files = []
    for root, _, fs in os.walk(path):
        for f in sorted(fs):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                files.append(os.path.join(root, f))
    if not files:
        raise FileNotFoundError(f'Không có ảnh tại: {path}')
    return sorted(files)

def _resize_keep_aspect(image, target_h, target_w):
    """Resize giữ aspect ratio — dùng area khi shrink, bicubic khi enlarge."""
    cur = tf.shape(image)[:2]
    h_ratio = tf.cast(target_h, tf.float64) / tf.cast(cur[0], tf.float64)
    w_ratio = tf.cast(target_w, tf.float64) / tf.cast(cur[1], tf.float64)
    ratio   = tf.minimum(h_ratio, w_ratio)
    new_h   = tf.cast(tf.round(tf.cast(cur[0], tf.float64) * ratio), tf.int32)
    new_w   = tf.cast(tf.round(tf.cast(cur[1], tf.float64) * ratio), tf.int32)
    new_size = tf.stack([new_h, new_w])
    shrinking = tf.logical_or(cur[0] > new_h, cur[1] > new_w)
    img4d = tf.expand_dims(image, 0)
    resized = tf.cond(
        shrinking,
        lambda: tf.image.resize(img4d, new_size, method='area'),
        lambda: tf.image.resize(img4d, new_size, method='bicubic')
    )
    return tf.clip_by_value(resized[0], 0.0, 255.0)

def _pad_to_target(image, target_h, target_w):
    """Padding đối xứng — 126 cho RGB, 0 cho saliency."""
    cur = tf.shape(image)
    is_rgb = tf.equal(cur[2], 3)
    pad_val = tf.cond(is_rgb, lambda: 126.0, lambda: 0.0)
    pad_v = tf.cast(target_h - cur[0], tf.float32) / 2.0
    pad_h = tf.cast(target_w - cur[1], tf.float32) / 2.0
    padding = [
        [tf.cast(tf.math.floor(pad_v), tf.int32), tf.cast(tf.math.ceil(pad_v), tf.int32)],
        [tf.cast(tf.math.floor(pad_h), tf.int32), tf.cast(tf.math.ceil(pad_h), tf.int32)],
        [0, 0],
    ]
    return tf.pad(image, padding, constant_values=pad_val)

def _parse(img_path, sal_path, h, w):
    """Load, resize (aspect-ratio), pad, normalize — rồi tạo 3 scale."""
    # ── Image ──────────────────────────────────────────────────────────
    img_raw = tf.io.read_file(img_path)
    img = tf.cond(tf.image.is_jpeg(img_raw),
                  lambda: tf.image.decode_jpeg(img_raw, channels=3),
                  lambda: tf.image.decode_png(img_raw,  channels=3))
    img = tf.cast(img, tf.float32)
    img = _resize_keep_aspect(img, h, w)
    img = _pad_to_target(img, h, w)
    img = img / 255.0
    img = tf.ensure_shape(img, [h, w, 3])

    # ── Saliency map ────────────────────────────────────────────────────
    sal_raw = tf.io.read_file(sal_path)
    sal = tf.cond(tf.image.is_jpeg(sal_raw),
                  lambda: tf.image.decode_jpeg(sal_raw, channels=1),
                  lambda: tf.image.decode_png(sal_raw,  channels=1))
    sal = tf.cast(sal, tf.float32)
    sal = _resize_keep_aspect(sal, h, w)
    sal = _pad_to_target(sal, h, w)
    sal = sal / 255.0
    sal = tf.ensure_shape(sal, [h, w, 1])

    # ── Tạo 3 scale inputs cho MSI ──────────────────────────────────────
    img_half    = tf.image.resize(img, [h//2, w//2], method='area')
    img_quarter = tf.image.resize(img, [h//4, w//4], method='area')

    return (img, img_half, img_quarter), sal

def build_dataset(img_dir, sal_dir, target_size, batch_size,
                  shuffle=True, buffer_size=1000):
    imgs = _get_file_list(img_dir)
    sals = _get_file_list(sal_dir)
    assert len(imgs) == len(sals), f'Mismatch: {len(imgs)} imgs vs {len(sals)} sals'
    h, w = target_size
    ds = tf.data.Dataset.from_tensor_slices((imgs, sals))
    if shuffle:
        ds = ds.shuffle(buffer_size, seed=42)
    ds = ds.map(lambda x, y: _parse(x, y, h, w),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, len(imgs)


In [ ]:
%%writefile /kaggle/working/metrics.py
"""
Evaluation metrics for visual saliency prediction.
Metrics: KLD, CC, NSS, SIM, AUC-Judd
All functions accept numpy arrays (H, W) normalised to [0,1].
"""
import numpy as np


def _normalize(x, eps=1e-7):
    s = x.sum()
    return x / (s + eps)


def kld_metric(gt, pred, eps=1e-7):
    """KL Divergence (lower is better)."""
    gt   = _normalize(gt.astype(np.float64))
    pred = _normalize(pred.astype(np.float64))
    return float(np.sum(gt * np.log(eps + gt / (eps + pred))))


def cc_metric(gt, pred, eps=1e-7):
    """Pearson Correlation Coefficient (higher is better, max 1)."""
    gt   = gt.astype(np.float64).ravel()
    pred = pred.astype(np.float64).ravel()
    gt   -= gt.mean();   pred -= pred.mean()
    denom = np.sqrt((gt**2).sum() * (pred**2).sum()) + eps
    return float(np.sum(gt * pred) / denom)


def sim_metric(gt, pred):
    """Similarity / histogram intersection (higher is better, max 1)."""
    gt   = _normalize(gt.astype(np.float64))
    pred = _normalize(pred.astype(np.float64))
    return float(np.sum(np.minimum(gt, pred)))


def nss_metric(gt_fixations, pred, eps=1e-7):
    """
    Normalized Scanpath Saliency (higher is better).
    gt_fixations: binary map (1 = fixation location).
    pred        : continuous saliency map.
    """
    pred = pred.astype(np.float64)
    pred = (pred - pred.mean()) / (pred.std() + eps)
    fix  = gt_fixations.astype(bool)
    if fix.sum() == 0:
        return 0.0
    return float(pred[fix].mean())


def auc_judd(gt_fixations, pred):
    """
    AUC-Judd (higher is better).
    Treats all fixation pixels as positives and random pixels as negatives.
    """
    pred  = pred.astype(np.float64).ravel()
    fix   = gt_fixations.astype(bool).ravel()
    if fix.sum() == 0 or (~fix).sum() == 0:
        return 0.5

    pos_scores = pred[fix]
    neg_scores = pred[~fix]

    thresholds = np.unique(pos_scores)[::-1]
    tprs, fprs = [], []
    for t in thresholds:
        tprs.append((pos_scores >= t).mean())
        fprs.append((neg_scores >= t).mean())
    tprs = np.array([0.0] + tprs + [1.0])
    fprs = np.array([0.0] + fprs + [1.0])
    try:
        return float(np.trapezoid(tprs, fprs))
    except AttributeError:
        return float(np.trapz(tprs, fprs))


def evaluate_batch(gt_maps, pred_maps, gt_fixations=None):
    """
    Compute all metrics over a batch.
    gt_maps, pred_maps : (N, H, W) numpy arrays in [0, 1]
    gt_fixations       : (N, H, W) binary numpy arrays (required for NSS, AUC)
    Returns dict with mean values.
    """
    results = {k: [] for k in ["KLD", "CC", "SIM", "NSS", "AUC"]}
    for i in range(len(gt_maps)):
        results["KLD"].append(kld_metric(gt_maps[i],  pred_maps[i]))
        results["CC"] .append(cc_metric (gt_maps[i],  pred_maps[i]))
        results["SIM"].append(sim_metric(gt_maps[i],  pred_maps[i]))
        if gt_fixations is not None:
            results["NSS"].append(nss_metric(gt_fixations[i], pred_maps[i]))
            results["AUC"].append(auc_judd  (gt_fixations[i], pred_maps[i]))
    return {k: float(np.mean(v)) for k, v in results.items() if v}


In [ ]:
%%writefile /kaggle/working/download_salicon.py
"""Tải SALICON dataset từ Google Drive."""
import os, zipfile
import gdown

def download_salicon(data_path):
    default_path   = os.path.join(data_path, "salicon")
    stimuli_train  = os.path.join(default_path, "stimuli",  "train")
    saliency_train = os.path.join(default_path, "saliency", "train")

    if os.path.exists(stimuli_train) and len(os.listdir(stimuli_train)) > 0:
        print("✅ SALICON đã tồn tại.")
        return default_path

    for d in [stimuli_train,
              os.path.join(default_path, "stimuli",  "val"),
              saliency_train,
              os.path.join(default_path, "saliency", "val"),
              os.path.join(default_path, "fixations")]:
        os.makedirs(d, exist_ok=True)

    tmp  = os.path.join(data_path, "tmp.zip")
    ids  = [
        "1g8j-hTT-51IG1UFwP0xTGhLdgIUCW5e5",  # stimuli
        "1P-jeZXCsjoKO79OhFUgnj6FGcyvmLDPj",  # fixations
        "1PnO7szbdub1559LfjYHMy65EDC4VhJC8",  # saliency maps
    ]
    paths = [
        os.path.join(default_path, "stimuli"),
        os.path.join(default_path, "fixations"),
        os.path.join(default_path, "saliency"),
    ]
    for i, fid in enumerate(ids):
        print(f"  Tải file {i+1}/3 ...", end="", flush=True)
        gdown.download(f"https://drive.google.com/uc?id={fid}&export=download", tmp, quiet=True)
        with zipfile.ZipFile(tmp, "r") as z:
            for f in z.namelist():
                if "test" not in f:
                    z.extract(f, paths[i])
        os.remove(tmp)
        print(" done!")

    images_dir = os.path.join(default_path, "stimuli", "images")
    if os.path.exists(images_dir):
        import shutil
        for sub in os.listdir(images_dir):
            shutil.move(os.path.join(images_dir, sub), os.path.join(default_path, "stimuli", sub))
        os.rmdir(images_dir)

    print("✅ SALICON tải xong!")
    return default_path


In [ ]:
%%writefile /kaggle/working/download_mit1003.py
"""
Tải MIT1003 dataset.
Cấu trúc sau khi giải nén:
  data/mit1003/stimuli/   — 1003 ảnh .jpeg
  data/mit1003/saliency/  — 1003 saliency maps .jpg  (gaussian-blurred)
  data/mit1003/fixations/ — 1003 fixation maps .jpg  (binary)
"""
import os, zipfile, urllib.request

MIT1003_STIMULI_URL  = "https://people.csail.mit.edu/tjudd/WherePeopleLook/ALLSTIMULI.zip"
MIT1003_FIXMAP_URL   = "https://people.csail.mit.edu/tjudd/WherePeopleLook/ALLFIXATIONMAPS.zip"
# Saliency (gaussian) maps — same server
MIT1003_SALMAP_URL   = "https://people.csail.mit.edu/tjudd/WherePeopleLook/ALLFIXATIONMAPS.zip"


def download_mit1003(data_path):
    root       = os.path.join(data_path, "mit1003")
    stim_dir   = os.path.join(root, "stimuli")
    sal_dir    = os.path.join(root, "saliency")
    fix_dir    = os.path.join(root, "fixations")

    n_imgs = sum(1 for _,_,fs in os.walk(stim_dir)
                for f in fs if f.lower().endswith((".jpg",".jpeg",".png")))
    if n_imgs >= 1000:
        print("✅ MIT1003 đã tồn tại, bỏ qua.")
        return root

    for d in [stim_dir, sal_dir, fix_dir]:
        os.makedirs(d, exist_ok=True)

    tmp = os.path.join(data_path, "tmp_mit.zip")

    def _dl(url, dest_dir, skip_suffix=None):
        print(f"  Tải {url.split('/')[-1]} ...", end="", flush=True)
        urllib.request.urlretrieve(url, tmp)
        with zipfile.ZipFile(tmp, "r") as z:
            for name in z.namelist():
                base = os.path.basename(name)
                if not base:
                    continue
                if skip_suffix and not base.lower().endswith(skip_suffix):
                    continue
                with z.open(name) as src, open(os.path.join(dest_dir, base), "wb") as dst:
                    dst.write(src.read())
        os.remove(tmp)
        print(" done!")

    _dl(MIT1003_STIMULI_URL, stim_dir, skip_suffix=(".jpeg", ".jpg", ".png"))
    _dl(MIT1003_FIXMAP_URL,  sal_dir,  skip_suffix=(".jpg",  ".jpeg", ".png"))
    print("✅ MIT1003 tải xong!")
    return root


In [ ]:
%%writefile /kaggle/working/download_cat2000.py
"""
Tải CAT2000 dataset.
Cấu trúc sau khi giải nén:
  data/cat2000/stimuli/   — 2000 ảnh (20 danh mục × 100 ảnh)
  data/cat2000/saliency/  — 2000 saliency maps  (gaussian-blurred)
  data/cat2000/fixations/ — 2000 fixation maps  (binary)
"""
import os, zipfile, time
import requests

CAT2000_URL = "http://saliency.mit.edu/trainSet.zip"


def download_cat2000(data_path):
    root     = os.path.join(data_path, "cat2000")
    stim_dir = os.path.join(root, "stimuli")

    if os.path.exists(stim_dir) and len(os.listdir(stim_dir)) >= 20:
        print("✅ CAT2000 đã tồn tại, bỏ qua.")
        return root

    os.makedirs(data_path, exist_ok=True)
    tmp = os.path.join(data_path, "tmp_cat.zip")

    # --- download với requests + retry + resume ---------------------------
    MAX_RETRIES = 5
    CHUNK       = 1024 * 1024   # 1 MB

    for attempt in range(1, MAX_RETRIES + 1):
        downloaded = os.path.getsize(tmp) if os.path.exists(tmp) else 0
        headers    = {"Range": f"bytes={downloaded}-"} if downloaded else {}
        try:
            print(f"  Tải CAT2000 (~1 GB) — lần {attempt} ", end="", flush=True)
            resp = requests.get(CAT2000_URL, headers=headers,
                                stream=True, timeout=(30, 120))
            # 206 = resume OK, 200 = server không hỗ trợ resume (restart)
            if resp.status_code == 200 and downloaded:
                downloaded = 0   # server gửi lại từ đầu
            if resp.status_code not in (200, 206):
                raise RuntimeError(f"HTTP {resp.status_code}")
            mode = "ab" if downloaded else "wb"
            with open(tmp, mode) as f:
                for chunk in resp.iter_content(CHUNK):
                    if chunk:
                        f.write(chunk)
                        print(".", end="", flush=True)
            print(" done!")
            break   # thành công
        except Exception as e:
            print(f" lỗi: {e}")
            if attempt < MAX_RETRIES:
                wait = 5 * attempt
                print(f"  Thử lại sau {wait}s...", flush=True)
                time.sleep(wait)
            else:
                raise RuntimeError("Không tải được CAT2000 sau nhiều lần thử.") from e

    # --- giải nén ---------------------------------------------------------
    print("  Đang giải nén...", end="", flush=True)
    with zipfile.ZipFile(tmp, "r") as z:
        for name in z.namelist():
            if not ("Output" in name or "allFixData" in name):
                z.extract(name, data_path)

    # --- rename thư mục ---------------------------------------------------
    train_path = os.path.join(data_path, "trainSet")
    if os.path.exists(train_path) and not os.path.exists(root):
        os.rename(train_path, root)

    for old, new in [("Stimuli",      "stimuli"),
                     ("FIXATIONLOCS", "fixations"),
                     ("FIXATIONMAPS", "saliency")]:
        src = os.path.join(root, old)
        dst = os.path.join(root, new)
        if os.path.exists(src) and not os.path.exists(dst):
            os.rename(src, dst)

    os.remove(tmp)
    print(" done!")
    print("✅ CAT2000 tải xong!")
    return root


## ── Bước 4: Tải dữ liệu ─────────────────────────────

In [ ]:
sys.path.insert(0, "/kaggle/working")
import numpy as np
from download_salicon  import download_salicon
from download_mit1003  import download_mit1003
from download_cat2000  import download_cat2000

salicon_root = download_salicon(DATA_DIR)
mit1003_root = download_mit1003(DATA_DIR)
cat2000_root = download_cat2000(DATA_DIR)

def _count_images(path):
    if not os.path.exists(path): return 0
    return sum(1 for _,_,fs in os.walk(path)
               for f in fs if f.lower().endswith((".jpg",".jpeg",".png")))

for split in ["train", "val"]:
    d = os.path.join(salicon_root, "stimuli", split)
    print(f"  SALICON {split}: {_count_images(d)} ảnh")
mit_stim = os.path.join(mit1003_root, "stimuli")
print(f"  MIT1003 : {_count_images(mit_stim)} ảnh")
cat_stim = os.path.join(cat2000_root, "stimuli")
n_cat = sum(len(fs) for _, _, fs in os.walk(cat_stim)) if os.path.exists(cat_stim) else 0
print(f"  CAT2000 : {n_cat} ảnh (20 danh mục)")


## ── Bước 5: Khởi tạo model ─────────────────────────

In [ ]:
import tensorflow as tf, numpy as np, importlib, sys
for mod in ['config', 'loss', 'model', 'data_loader', 'metrics']:
    sys.modules.pop(mod, None)

import config, loss as loss_module
from model       import build_msinet, load_vgg16_hybrid_weights
from data_loader import build_dataset

gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

TARGET_SIZE = config.DIMS['image_size_salicon']  # (240, 320)
BATCH_SIZE  = config.PARAMS['batch_size']
N_EPOCHS    = config.PARAMS['n_epochs']
LR          = config.PARAMS['learning_rate']

train_ds, n_train = build_dataset(
    os.path.join(salicon_root, 'stimuli',  'train'),
    os.path.join(salicon_root, 'saliency', 'train'),
    TARGET_SIZE, BATCH_SIZE, shuffle=True)
valid_ds, n_valid = build_dataset(
    os.path.join(salicon_root, 'stimuli',  'val'),
    os.path.join(salicon_root, 'saliency', 'val'),
    TARGET_SIZE, BATCH_SIZE, shuffle=False)

print(f'Train: {n_train} | Val: {n_valid}')

model = build_msinet(input_shape=(*TARGET_SIZE, 3))
model.summary(line_length=90)

# ── Load pretrained VGG16-Hybrid weights từ checkpoint ───────────────────
CKPT_PREFIX = os.path.join(WEIGHTS_DIR, 'vgg16_hybrid.ckpt')
CKPT_INDEX  = CKPT_PREFIX + '.index'

if not os.path.exists(CKPT_INDEX):
    print('  Đang tải VGG16-Hybrid weights...')
    import gdown, zipfile
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    zip_path = os.path.join(WEIGHTS_DIR, 'tmp_vgg.zip')
    gdown.download(
        'https://drive.google.com/uc?id=1ff0va472Xs1bvidCwRlW3Ctf7Hbyyn7p&export=download',
        zip_path, quiet=False)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(WEIGHTS_DIR)
    os.remove(zip_path)

# Đọc weights từ checkpoint bằng raw reader — không đụng Keras tracking
try:
    reader   = tf.train.load_checkpoint(CKPT_PREFIX)
    var_map  = reader.get_variable_to_shape_map()
    VGG_CONV = [
        'conv1_1','conv1_2','conv2_1','conv2_2',
        'conv3_1','conv3_2','conv3_3',
        'conv4_1','conv4_2','conv4_3',
        'conv5_1','conv5_2','conv5_3',
    ]
    copied = 0
    for base in VGG_CONV:
        # Tìm key kernel/bias khớp với tên layer base (không lấy _half/_quarter)
        k_key = next((k for k in var_map
                      if f'/{base}/' in k or k.startswith(base+'/')
                      and 'kernel' in k), None)
        b_key = next((k for k in var_map
                      if f'/{base}/' in k or k.startswith(base+'/')
                      and 'bias' in k), None)
        if k_key is None:
            continue
        kernel = reader.get_tensor(k_key)
        bias   = reader.get_tensor(b_key) if b_key else None
        for sfx in ['', '_half', '_quarter']:
            try:
                layer = model.get_layer(base + sfx)
                layer.set_weights([kernel, bias] if bias is not None else [kernel])
                copied += 1
            except (ValueError, KeyError):
                pass
    print(f'✅ Loaded pretrained weights vào {copied} conv layers')
except Exception as e:
    print(f'⚠️  Không load được checkpoint: {e} — train từ random init')

def kld_loss(y_true, y_pred): return loss_module.kld(y_true, y_pred)
model.compile(optimizer=tf.keras.optimizers.Adam(LR), loss=kld_loss)
print('✅ Model compiled')

## ── Bước 6: Huấn luyện trên SALICON ────────────────

In [ ]:
CKPT_DIR    = os.path.join(RESULTS_DIR, 'ckpts')
BEST_CKPT   = os.path.join(CKPT_DIR, 'best', 'msinet_salicon_best.weights.h5')
HISTORY_DIR = os.path.join(RESULTS_DIR, 'history')

# ── Wrapper dataset: (img_full, img_half, img_quarter) → dict input ────
def to_model_inputs(scales, sal):
    img_full, img_half, img_quarter = scales
    return ({'input_full': img_full,
             'input_half': img_half,
             'input_quarter': img_quarter}, sal)

train_ds_mi = train_ds.map(to_model_inputs)
valid_ds_mi = valid_ds.map(to_model_inputs)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_CKPT, save_weights_only=True, monitor='val_loss',
        mode='min', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.CSVLogger(
        os.path.join(HISTORY_DIR, 'training_log.csv'), append=True),
]

print('🚀 Bắt đầu training MSI-Net trên SALICON...\n')
history = model.fit(train_ds_mi, epochs=N_EPOCHS,
                    validation_data=valid_ds_mi, callbacks=callbacks, verbose=1)
print('\n✅ Training xong!')


In [ ]:
# ── Vẽ loss curve ngay sau khi train SALICON ──────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("MSI-Net Baseline — Training SALICON", fontsize=13, fontweight="bold")

# Train vs Val Loss
axes[0].plot(history.history["loss"],     label="Train KLD", linewidth=2, marker="o", markersize=4)
axes[0].plot(history.history["val_loss"], label="Val KLD",   linewidth=2, marker="s", markersize=4)
axes[0].set_title("KLD Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("KLD Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Train/Val gap (overfitting check)
gap = [v - t for t, v in zip(history.history["loss"], history.history["val_loss"])]
axes[1].plot(gap, color="tomato", linewidth=2, marker="^", markersize=4)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Val − Train Gap (overfit nếu > 0)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Gap")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(HISTORY_DIR, "salicon_train_loss.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Best val_loss: {min(history.history['val_loss']):.4f} "
      f"(epoch {history.history['val_loss'].index(min(history.history['val_loss']))+1})")

## ── Bước 7: Đánh giá trên SALICON val ──────────────

In [ ]:
def evaluate_model(model, dataset, n_batches=None, dataset_name=''):
    """Tính KLD, CC, SIM, NSS, AUC-Judd — hỗ trợ cả single và multi-input."""
    from metrics import evaluate_batch
    all_gt, all_pred = [], []

    for i, (inputs, sals) in enumerate(dataset):
        if n_batches and i >= n_batches:
            break
        # Multi-input: inputs là tuple (full, half, quarter) hoặc dict
        if isinstance(inputs, (tuple, list)):
            img_full, img_half, img_quarter = inputs
            preds = model.predict(
                {'input_full': img_full, 'input_half': img_half,
                 'input_quarter': img_quarter}, verbose=0)
        elif isinstance(inputs, dict):
            preds = model.predict(inputs, verbose=0)
        else:
            preds = model.predict(inputs, verbose=0)

        if isinstance(preds, (list, tuple)): preds = preds[0]
        gt_np   = sals.numpy()[:, :, :, 0]
        pred_np = preds[:, :, :, 0]
        all_gt.append(gt_np); all_pred.append(pred_np)

    all_gt   = np.concatenate(all_gt,   axis=0)
    all_pred = np.concatenate(all_pred, axis=0)
    fix_maps = (all_gt > 0.1).astype(np.float32)
    scores   = evaluate_batch(all_gt, all_pred, gt_fixations=fix_maps)

    print(f"\n{'='*55}")
    print(f'📊 Evaluation — {dataset_name}')
    print(f"{'='*55}")
    for k, v in scores.items():
        arrow = '↓ better' if k == 'KLD' else '↑ better'
        print(f'  {k:<8}: {v:.4f}  ({arrow})')
    print(f"{'='*55}")
    return scores

if os.path.exists(BEST_CKPT):
    model.load_weights(BEST_CKPT)
    print('✅ Loaded best weights')

scores_salicon = evaluate_model(model, valid_ds, dataset_name='SALICON val')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_predictions(model, dataset, dataset_name, n_samples=5, save_dir=None):
    """Hiển thị n_samples ảnh: Original | Predicted | Ground Truth — hỗ trợ multi-input."""
    imgs_list, sals_list, preds_list = [], [], []

    for inputs, sals in dataset:
        if isinstance(inputs, (tuple, list)):
            img_full, img_half, img_quarter = inputs
            preds = model.predict(
                {'input_full': img_full, 'input_half': img_half,
                 'input_quarter': img_quarter}, verbose=0)
            imgs_list.append(img_full.numpy())
        elif isinstance(inputs, dict):
            preds = model.predict(inputs, verbose=0)
            imgs_list.append(list(inputs.values())[0].numpy())
        else:
            preds = model.predict(inputs, verbose=0)
            imgs_list.append(inputs.numpy())
        sals_list.append(sals.numpy())
        preds_list.append(preds)
        if sum(len(x) for x in imgs_list) >= n_samples:
            break

    imgs_all  = np.concatenate(imgs_list,  axis=0)[:n_samples]
    sals_all  = np.concatenate(sals_list,  axis=0)[:n_samples]
    preds_all = np.concatenate(preds_list, axis=0)[:n_samples]

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    fig.suptitle(f'Kết quả dự đoán — {dataset_name}', fontsize=14, fontweight='bold')
    for col, title in enumerate(['Ảnh gốc', 'Predicted Saliency', 'Ground Truth']):
        axes[0, col].set_title(title, fontsize=12, fontweight='bold')
    for i in range(n_samples):
        axes[i, 0].imshow(imgs_all[i]); axes[i, 0].axis('off')
        pred = preds_all[i, :, :, 0]
        pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-7)
        axes[i, 1].imshow(pred, cmap='jet'); axes[i, 1].axis('off')
        axes[i, 2].imshow(sals_all[i, :, :, 0], cmap='jet'); axes[i, 2].axis('off')
    plt.tight_layout()
    if save_dir:
        path = os.path.join(save_dir, f"viz_{dataset_name.replace(' ', '_')}.png")
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'✅ Đã lưu: {path}')
    plt.show()

visualize_predictions(model, valid_ds, dataset_name='SALICON val',
                      n_samples=5, save_dir=RESULTS_DIR)


## ── Bước 8: Fine-tune trên MIT1003 ─────────────────


In [ ]:
import os, shutil, random

mit1003_root = '/kaggle/working/data/mit1003'
sal_src    = os.path.join(mit1003_root, 'saliency')
stim_src   = os.path.join(mit1003_root, 'stimuli')
stim_train = os.path.join(mit1003_root, 'stimuli',  'train')
stim_val   = os.path.join(mit1003_root, 'stimuli',  'val')
sal_train  = os.path.join(mit1003_root, 'saliency', 'train')
sal_val    = os.path.join(mit1003_root, 'saliency', 'val')

# Tạo thư mục trước
for d in [stim_train, stim_val, sal_train, sal_val]:
    os.makedirs(d, exist_ok=True)

# Build map: stem -> fixMap file
sal_map = {}
for f in os.listdir(sal_src):
    if '_fixMap' in f and f.lower().endswith(('.png', '.jpg', '.jpeg')):
        stem = f.replace('_fixMap.jpg', '').replace('_fixMap.png', '')
        sal_map[stem] = os.path.join(sal_src, f)
print(f'Tìm thấy {len(sal_map)} fixMap files')

# Lấy danh sách ảnh gốc (flat trong stim_src, bỏ qua train/ val/)
all_imgs = sorted([f for f in os.listdir(stim_src)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                   and os.path.isfile(os.path.join(stim_src, f))])

if not all_imgs:
    # Đã split rồi — gom lại từ train/ val/
    all_imgs = []
    for sp in ['train', 'val']:
        sp_dir = os.path.join(stim_src, sp)
        if os.path.isdir(sp_dir):
            all_imgs += [f for f in os.listdir(sp_dir)
                         if f.lower().endswith(('.jpg','.jpeg','.png'))]
    all_imgs = sorted(all_imgs)

print(f'Tổng ảnh MIT1003: {len(all_imgs)}')

# Split 90/10
random.seed(42); random.shuffle(all_imgs)
n_tr = int(len(all_imgs) * 0.9)
split_map = {f: 'train' for f in all_imgs[:n_tr]}
split_map.update({f: 'val' for f in all_imgs[n_tr:]})

def copy_stim_sal(fn, split):
    # Tìm ảnh gốc
    src_img = os.path.join(stim_src, fn)
    if not os.path.exists(src_img):
        for sp2 in ['train', 'val']:
            c = os.path.join(stim_src, sp2, fn)
            if os.path.exists(c): src_img = c; break
    dst_img = os.path.join(mit1003_root, 'stimuli', split, fn)
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        shutil.copy2(src_img, dst_img)

    # Tìm saliency
    stem = os.path.splitext(fn)[0]
    if stem in sal_map:
        dst_sal = os.path.join(mit1003_root, 'saliency', split,
                               os.path.basename(sal_map[stem]))
        if not os.path.exists(dst_sal):
            shutil.copy2(sal_map[stem], dst_sal)
        return True
    return False

tr_ok = tr_miss = v_ok = v_miss = 0
for fn, split in split_map.items():
    found = copy_stim_sal(fn, split)
    if split == 'train':
        tr_ok += found; tr_miss += not found
    else:
        v_ok  += found; v_miss  += not found

print(f'Train — ảnh: {n_tr},  saliency matched: {tr_ok}, missing: {tr_miss}')
print(f'Val   — ảnh: {len(all_imgs)-n_tr}, saliency matched: {v_ok},  missing: {v_miss}')


In [ ]:
import random, shutil, importlib

import model as model_module
importlib.reload(model_module)
from model import build_msinet, load_vgg16_hybrid_weights

def _clone_weights(target_model, ckpt_path, src_input_shape):
    """Copy weights từ checkpoint sang model target (theo tên layer)."""
    src_model = build_msinet(input_shape=src_input_shape)
    src_model.load_weights(ckpt_path)
    src_map = {l.name: l for l in src_model.layers}
    copied = 0
    for layer in target_model.layers:
        if layer.name not in src_map: continue
        w = src_map[layer.name].get_weights()
        if not w: continue
        try: layer.set_weights(w); copied += 1
        except ValueError: pass
    del src_model
    return copied

MIT_SIZE = config.DIMS['image_size_mit1003']  # (360, 360)
mit_stim = os.path.join(mit1003_root, 'stimuli')
mit_sal  = os.path.join(mit1003_root, 'saliency')

# Đọc ảnh trực tiếp trong mit_stim (không đệ quy vào train/val nếu đã tồn tại)
_mit_imgs_flat = [f for f in os.listdir(mit_stim)
                  if f.lower().endswith(('.jpg','.jpeg','.png'))]
if not _mit_imgs_flat:
    # Nếu đã split rồi thì gom lại từ train+val
    _mit_imgs_flat = []
    for sp in ['train', 'val']:
        sp_dir = os.path.join(mit_stim, sp)
        if os.path.isdir(sp_dir):
            _mit_imgs_flat += [f for f in os.listdir(sp_dir)
                               if f.lower().endswith(('.jpg','.jpeg','.png'))]
if len(_mit_imgs_flat) >= 100:
    all_imgs = sorted(_mit_imgs_flat)
    random.seed(42); random.shuffle(all_imgs)
    n_train_mit = int(len(all_imgs) * 0.9)
    train_imgs  = all_imgs[:n_train_mit]
    val_imgs    = all_imgs[n_train_mit:]

    for split, lst in [('train', train_imgs), ('val', val_imgs)]:
        for sub in ['stimuli', 'saliency']:
            os.makedirs(os.path.join(mit1003_root, sub, split), exist_ok=True)
        for fn in lst:
            # Tìm ảnh gốc: có thể ở mit_stim/ hoặc đã trong train/val/
            src_s = os.path.join(mit_stim, fn)
            if not os.path.exists(src_s):
                for sp2 in ['train', 'val']:
                    candidate = os.path.join(mit_stim, sp2, fn)
                    if os.path.exists(candidate):
                        src_s = candidate; break
            dst_s = os.path.join(mit1003_root, 'stimuli', split, fn)
            if os.path.exists(src_s) and not os.path.exists(dst_s):
                shutil.copy2(src_s, dst_s)
            stem   = os.path.splitext(fn)[0]
            sal_fn = f'{stem}_fixMap.jpg'
            src_sal = os.path.join(mit_sal, sal_fn)
            if not os.path.exists(src_sal):
                sal_fn  = f'{stem}_fixMap.png'
                src_sal = os.path.join(mit_sal, sal_fn)
            dst_sal = os.path.join(mit1003_root, 'saliency', split, sal_fn)
            if os.path.exists(src_sal) and not os.path.exists(dst_sal):
                shutil.copy2(src_sal, dst_sal)

    mit_train_ds, n_mt = build_dataset(
        os.path.join(mit1003_root, 'stimuli',  'train'),
        os.path.join(mit1003_root, 'saliency', 'train'),
        MIT_SIZE, BATCH_SIZE, shuffle=True)
    mit_val_ds, n_mv = build_dataset(
        os.path.join(mit1003_root, 'stimuli',  'val'),
        os.path.join(mit1003_root, 'saliency', 'val'),
        MIT_SIZE, BATCH_SIZE, shuffle=False)
    print(f'MIT1003 — train: {n_mt} | val: {n_mv}')

    mit_model = build_msinet(input_shape=(*MIT_SIZE, 3))
    if os.path.exists(BEST_CKPT):
        copied = _clone_weights(mit_model, BEST_CKPT, src_input_shape=(240,320,3))
        print(f'  ✅ Copied {copied} layers từ SALICON → MIT1003 model')
    else:
        print('  ⚠️  Không tìm thấy SALICON checkpoint — train từ scratch')
        load_vgg16_hybrid_weights(mit_model, VGG_WEIGHTS)

    mit_model.compile(optimizer=tf.keras.optimizers.Adam(LR * 0.1), loss=kld_loss)

    # Wrapper multi-input
    def to_mi(scales, sal):
        f, h, q = scales
        return ({'input_full': f, 'input_half': h, 'input_quarter': q}, sal)
    mit_train_mi = mit_train_ds.map(to_mi)
    mit_val_mi   = mit_val_ds.map(to_mi)

    MIT_CKPT = os.path.join(CKPT_DIR, 'best', 'msinet_mit1003_best.weights.h5')
    print('\n🔁 Fine-tuning trên MIT1003 (từ SALICON weights)...')
    mit_ft_history = mit_model.fit(
        mit_train_mi, epochs=10, validation_data=mit_val_mi,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                MIT_CKPT, save_weights_only=True, monitor='val_loss',
                mode='min', save_best_only=True, verbose=1),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ], verbose=1)
    if os.path.exists(MIT_CKPT):
        mit_model.load_weights(MIT_CKPT)
        print(f'✅ MIT1003 model saved → {MIT_CKPT}')
else:
    print('⚠️  MIT1003 chưa tải được — bỏ qua fine-tune')
    mit_train_ds = mit_val_ds = mit_ft_history = mit_model = None
    MIT_CKPT = ''


In [ ]:
# ── Vẽ loss curve ngay sau fine-tune MIT1003 ──────────────────────────
import matplotlib.pyplot as plt

if mit_ft_history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("MSI-Net Fine-tune — MIT1003", fontsize=13, fontweight="bold")

    axes[0].plot(mit_ft_history.history["loss"],     label="Train KLD", linewidth=2, marker="o", markersize=4)
    axes[0].plot(mit_ft_history.history["val_loss"], label="Val KLD",   linewidth=2, marker="s", markersize=4)
    axes[0].set_title("KLD Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("KLD Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    gap = [v - t for t, v in zip(mit_ft_history.history["loss"], mit_ft_history.history["val_loss"])]
    axes[1].plot(gap, color="tomato", linewidth=2, marker="^", markersize=4)
    axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("Val − Train Gap (overfit nếu > 0)")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Gap")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(HISTORY_DIR, "mit1003_finetune_loss_inline.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Best val_loss: {min(mit_ft_history.history['val_loss']):.4f} "
          f"(epoch {mit_ft_history.history['val_loss'].index(min(mit_ft_history.history['val_loss']))+1})")
else:
    print("⚠️  Không có history MIT1003 để vẽ")

## ── Bước 9: Đánh giá MIT1003 val ──────────────────────


In [ ]:
import matplotlib.pyplot as plt

# ── Đánh giá MIT1003 val ──────────────────────────────────────────
if mit_ft_history is not None:
    val_ds_mit = mit_v_ds if False else mit_val_ds
    scores_mit = evaluate_model(mit_model, val_ds_mit, dataset_name="MIT1003 val")

    # Loss curve fine-tune MIT1003
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(mit_ft_history.history["loss"],     label="Train Loss", linewidth=2)
    ax.plot(mit_ft_history.history["val_loss"], label="Val Loss",   linewidth=2)
    ax.set_title("Fine-tune MIT1003 — Loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(HISTORY_DIR, "mit1003_finetune_loss.png"), dpi=150)
    plt.show()
else:
    scores_mit = {}


In [ ]:
# ── Test MIT1003 val ──
visualize_predictions(mit_model, mit_val_ds, dataset_name="MIT1003 val",
                      n_samples=5, save_dir=RESULTS_DIR)

## ── Bước 10: Fine-tune trên CAT2000 ───────────────────


In [ ]:
import random, shutil
from collections import defaultdict

CAT_SIZE = config.DIMS['image_size_cat2000']  # (216, 384)
cat_stim = os.path.join(cat2000_root, 'stimuli')
cat_sal  = os.path.join(cat2000_root, 'saliency')

if os.path.exists(cat_stim) and sum(len(f) for _,_,f in os.walk(cat_stim)) >= 100:
    all_cat_imgs = []
    for subdir in sorted(os.listdir(cat_stim)):
        sub_path = os.path.join(cat_stim, subdir)
        if os.path.isdir(sub_path):
            for fn in sorted(os.listdir(sub_path)):
                if fn.lower().endswith(('.jpg','.jpeg','.png')):
                    all_cat_imgs.append((subdir, fn))

    random.seed(42)
    by_cat = defaultdict(list)
    for cat, fn in all_cat_imgs: by_cat[cat].append(fn)
    train_cat, val_cat = [], []
    for cat, fns in by_cat.items():
        random.shuffle(fns)
        n_tr = int(len(fns) * 0.8)
        for fn in fns[:n_tr]: train_cat.append((cat, fn))
        for fn in fns[n_tr:]: val_cat.append((cat, fn))
    print(f'CAT2000 — train: {len(train_cat)} | val: {len(val_cat)}')

    for split, lst in [('train', train_cat), ('val', val_cat)]:
        for sub in ['stimuli', 'saliency']:
            os.makedirs(os.path.join(cat2000_root, sub, split), exist_ok=True)
        for cat, fn in lst:
            src_img = os.path.join(cat_stim, cat, fn)
            dst_img = os.path.join(cat2000_root, 'stimuli', split, f'{cat}_{fn}')
            if os.path.exists(src_img) and not os.path.exists(dst_img):
                shutil.copy2(src_img, dst_img)
            for ext in [fn, fn.replace('.jpeg','.jpg'), fn.replace('.png','.jpg')]:
                src_sal = os.path.join(cat_sal, cat, ext)
                if os.path.exists(src_sal):
                    dst_sal = os.path.join(cat2000_root, 'saliency', split, f'{cat}_{ext}')
                    if not os.path.exists(dst_sal): shutil.copy2(src_sal, dst_sal)
                    break

    cat_tr_ds, n_ct = build_dataset(
        os.path.join(cat2000_root, 'stimuli',  'train'),
        os.path.join(cat2000_root, 'saliency', 'train'),
        CAT_SIZE, BATCH_SIZE, shuffle=True)
    cat_v_ds, n_cv = build_dataset(
        os.path.join(cat2000_root, 'stimuli',  'val'),
        os.path.join(cat2000_root, 'saliency', 'val'),
        CAT_SIZE, BATCH_SIZE, shuffle=False)
    print(f'  Dataset: train={n_ct} | val={n_cv}')

    cat_model = build_msinet(input_shape=(*CAT_SIZE, 3))
    if os.path.exists(MIT_CKPT):
        copied = _clone_weights(cat_model, MIT_CKPT, src_input_shape=(*MIT_SIZE, 3))
        print(f'  ✅ Copied {copied} layers từ MIT1003 → CAT2000 model')
    elif os.path.exists(BEST_CKPT):
        copied = _clone_weights(cat_model, BEST_CKPT, src_input_shape=(240,320,3))
        print(f'  ⚠️  Fallback: copied {copied} layers từ SALICON → CAT2000 model')
    else:
        print('  ⚠️  Không tìm thấy checkpoint — load VGG16 pretrained')
        load_vgg16_hybrid_weights(cat_model, VGG_WEIGHTS)

    cat_model.compile(optimizer=tf.keras.optimizers.Adam(LR * 0.1), loss=kld_loss)

    def to_mi(scales, sal):
        f, h, q = scales
        return ({'input_full': f, 'input_half': h, 'input_quarter': q}, sal)
    cat_tr_mi = cat_tr_ds.map(to_mi)
    cat_v_mi  = cat_v_ds.map(to_mi)

    CAT_CKPT = os.path.join(CKPT_DIR, 'best', 'msinet_cat2000_best.weights.h5')
    print('\n🔁 Fine-tuning trên CAT2000 (từ MIT1003 weights)...')
    cat_ft_history = cat_model.fit(
        cat_tr_mi, epochs=10, validation_data=cat_v_mi,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                CAT_CKPT, save_weights_only=True, monitor='val_loss',
                mode='min', save_best_only=True, verbose=1),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ], verbose=1)
    if os.path.exists(CAT_CKPT):
        cat_model.load_weights(CAT_CKPT)
        print(f'✅ CAT2000 model saved → {CAT_CKPT}')
else:
    print('⚠️  CAT2000 chưa tải được — bỏ qua fine-tune')
    cat_tr_ds = cat_v_ds = cat_ft_history = cat_model = None
    CAT_CKPT = ''


In [ ]:
# ── Vẽ loss curve ngay sau fine-tune CAT2000 ──────────────────────────
import matplotlib.pyplot as plt

if cat_ft_history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("MSI-Net Fine-tune — CAT2000", fontsize=13, fontweight="bold")

    axes[0].plot(cat_ft_history.history["loss"],     label="Train KLD", linewidth=2, marker="o", markersize=4)
    axes[0].plot(cat_ft_history.history["val_loss"], label="Val KLD",   linewidth=2, marker="s", markersize=4)
    axes[0].set_title("KLD Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("KLD Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    gap = [v - t for t, v in zip(cat_ft_history.history["loss"], cat_ft_history.history["val_loss"])]
    axes[1].plot(gap, color="tomato", linewidth=2, marker="^", markersize=4)
    axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("Val − Train Gap (overfit nếu > 0)")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Gap")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(HISTORY_DIR, "cat2000_finetune_loss_inline.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Best val_loss: {min(cat_ft_history.history['val_loss']):.4f} "
          f"(epoch {cat_ft_history.history['val_loss'].index(min(cat_ft_history.history['val_loss']))+1})")
else:
    print("⚠️  Không có history CAT2000 để vẽ")

## ── Bước 11: Đánh giá CAT2000 val ─────────────────────


In [ ]:
import matplotlib.pyplot as plt

# ── Đánh giá CAT2000 val ─────────────────────────────────────────
if cat_ft_history is not None:
    scores_cat = evaluate_model(cat_model, cat_v_ds, dataset_name="CAT2000 val")

    # Loss curve fine-tune CAT2000
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(cat_ft_history.history["loss"],     label="Train Loss", linewidth=2)
    ax.plot(cat_ft_history.history["val_loss"], label="Val Loss",   linewidth=2)
    ax.set_title("Fine-tune CAT2000 — Loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(HISTORY_DIR, "cat2000_finetune_loss.png"), dpi=150)
    plt.show()
else:
    scores_cat = {}


In [ ]:
# ── Test CAT2000 val ──
visualize_predictions(cat_model, cat_v_ds, dataset_name="CAT2000 val",
                      n_samples=5, save_dir=RESULTS_DIR)

## ── Bước 12: Loss curve SALICON ──────────────────────────────

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
log_path = os.path.join(HISTORY_DIR, "training_log.csv")
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(df["loss"],     label="Train KLD", linewidth=2)
    ax.plot(df["val_loss"], label="Val KLD",   linewidth=2)
    ax.set_title("MSI-Net Baseline — KLD Loss (SALICON)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("KLD Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(HISTORY_DIR, "loss_curve.png"), dpi=150)
    plt.show()

## ── Bước 13: Inference — ảnh mẫu ───────────────────

In [ ]:
# ── Inference ảnh mẫu từ SALICON val ─────────────────────────────────
scales, sample_sals = next(iter(valid_ds))
img_full, img_half, img_quarter = scales
img_full    = img_full[:4];    img_half    = img_half[:4]
img_quarter = img_quarter[:4]; sample_sals = sample_sals[:4]

preds = model.predict({'input_full': img_full,
                        'input_half': img_half,
                        'input_quarter': img_quarter}, verbose=0)
if isinstance(preds, (list, tuple)): preds = preds[0]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
for i in range(4):
    axes[i,0].imshow(img_full[i].numpy()); axes[i,0].axis('off')
    axes[i,1].imshow(sample_sals[i,:,:,0].numpy(), cmap='jet'); axes[i,1].axis('off')
    axes[i,2].imshow(preds[i,:,:,0], cmap='jet'); axes[i,2].axis('off')
    if i == 0:
        axes[i,0].set_title('Input'); axes[i,1].set_title('GT'); axes[i,2].set_title('Pred')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'images', 'msinet_predictions.png'), dpi=150)
plt.show()


## ── Bước 14: Bảng kết quả tổng hợp ─────────────────

In [ ]:
print("\n" + "="*70)
print("📋 KẾT QUẢ ĐÁNH GIÁ — MSI-Net Baseline")
print("="*70)
header = f"{'Metric':<10} {'SALICON val':>14} {'MIT1003 val':>14} {'CAT2000 val':>14}"
print(header)
print("-"*70)
for k in ["KLD", "CC", "SIM", "NSS", "AUC"]:
    s1 = f"{scores_salicon.get(k, float('nan')):.4f}"
    s2 = f"{scores_mit.get(k, float('nan')):.4f}" if scores_mit else "  N/A"
    s3 = f"{scores_cat.get(k, float('nan')):.4f}" if scores_cat else "  N/A"
    print(f"{k:<10} {s1:>14} {s2:>14} {s3:>14}")
print("="*70)
print("Note: KLD ↓ better | CC, SIM, NSS, AUC ↑ better")


## ── Bước 15: Lưu model & zip kết quả ───────────────

In [ ]:
model.save(os.path.join(RESULTS_DIR, "baseline_final.keras"))
model.save_weights(os.path.join(RESULTS_DIR, "baseline_weights.weights.h5"))
print("✅ Model saved")

import zipfile
zip_path = "/kaggle/working/baseline_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(RESULTS_DIR):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, "/kaggle/working"))
print(f"✅ Đã nén: {zip_path}")
from IPython.display import FileLink
FileLink(zip_path)

In [ ]:
## ── Bước 16: Đóng gói toàn bộ output ───────────────────────────────
import os, zipfile, datetime
from IPython.display import FileLink, display

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path  = f"/kaggle/working/msinet_full_{timestamp}.zip"

# ── Danh sách thư mục & file cần đóng gói ───────────────────────────
INCLUDE_DIRS = [
    RESULTS_DIR,          # ckpts, history (loss curves), images (viz)
    WEIGHTS_DIR,          # weights tải về (nếu có)
]

INCLUDE_FILES = [
    # Module .py
    "/kaggle/working/config.py",
    "/kaggle/working/loss.py",
    "/kaggle/working/model.py",
    "/kaggle/working/data_loader.py",
    "/kaggle/working/metrics.py",
    "/kaggle/working/download_salicon.py",
    "/kaggle/working/download_mit1003.py",
    "/kaggle/working/download_cat2000.py",
]

added, skipped = [], []

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

    # Đóng gói toàn bộ thư mục
    for base_dir in INCLUDE_DIRS:
        if not os.path.exists(base_dir):
            skipped.append(base_dir); continue
        for root, _, files in os.walk(base_dir):
            for fn in files:
                fp  = os.path.join(root, fn)
                arc = os.path.relpath(fp, "/kaggle/working")
                zf.write(fp, arc)
                added.append(arc)

    # Đóng gói các file lẻ
    for fp in INCLUDE_FILES:
        if os.path.exists(fp):
            arc = os.path.relpath(fp, "/kaggle/working")
            zf.write(fp, arc)
            added.append(arc)
        else:
            skipped.append(fp)

# ── In tóm tắt ──────────────────────────────────────────────────────
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"{'='*55}")
print(f"📦 Đóng gói hoàn tất: {os.path.basename(zip_path)}")
print(f"   Kích thước : {size_mb:.2f} MB")
print(f"   Số file    : {len(added)}")
print(f"{'='*55}")

# Liệt kê theo nhóm
groups = {"weights/": [], "history/": [], "images/": [], "ckpts/": [], "other": []}
for a in added:
    matched = False
    for key in groups:
        if key in a:
            groups[key].append(a); matched = True; break
    if not matched:
        groups["other"].append(a)
for grp, files in groups.items():
    if files:
        print(f"  [{grp}] {len(files)} file(s)")
        for f in files: print(f"      {f}")

if skipped:
    print(f"\n  ⚠️  Bỏ qua (không tồn tại): {len(skipped)}")
    for s in skipped: print(f"      {s}")

print(f"{'='*55}")
display(FileLink(zip_path, result_html_prefix="⬇️  Tải về: "))

In [ ]:
from IPython.display import FileLink
FileLink(r'msinet_full_20260606_063730.zip')